In [2]:
#!/usr/bin/env python3
r"""
Copy ONE DLC tracking folder per session (day) into a per-mouse folder.

For every session that has a skeleton tracking result, pick skeleton OR
transformer using the flag files in the session (Day) folder, then COPY the
chosen tracking folder into  DEST_ROOT / <mouseID> / .  Originals are never
moved or modified.

Selection rule (mirrors your snippet), evaluated per session/Day folder:
    - flag1.txt present  ................................ use SKELETON
    - neither flag.txt nor flag1.txt present ............ use SKELETON
    - flag.txt present AND flag1.txt absent ............. use TRANSFORMER

Run it once with DRY_RUN = True (default) to review every planned copy, then
set DRY_RUN = False to actually copy.
"""

import re
import shutil
import sys
from pathlib import Path

# ============================================================================
# CONFIG
# ============================================================================

SOURCE_ROOT = r"C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\RI_datasets\trainingSessions"

DEST_ROOT = r"C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions_topView_DLC_multiAnimal\trainingSessions_topView_DLC_multiAnimal"

PCUTOFF = 0.8

# Filenames that identify the raw h5 in each tracking folder.
SKELETON_H5_SUFFIX = "snapshot_best-100_sk.h5"       # in *_skeleton folders
TRANSFORMER_H5_SUFFIX = "snapshot_best-100_sk_tr.h5"  # in *_transformer folders

# Destination layout under each mouseID folder:
#   "merged"  -> DEST/<mouseID>/topView_DLCtracking_pcutoff_0.8_<method>/
#                (all chosen days merge into one folder per method; the DLC
#                 filenames already contain _Day0X so they don't collide)
#   "per_day" -> DEST/<mouseID>/<Day>/topView_DLCtracking_pcutoff_0.8_<method>/
LAYOUT = "merged"

# Skip copying the (often large) plot-poses folders you said you don't need.
EXCLUDE_PLOT_POSES = True

# Safety: preview first. Set to False to actually copy.
DRY_RUN = False

# ============================================================================
# END CONFIG
# ============================================================================

_IGNORE = shutil.ignore_patterns("plot-poses", "plot_poses") if EXCLUDE_PLOT_POSES else None


def mouse_id_from(h5_path: Path, skeleton_folder: Path) -> str:
    """mouseID from the h5 filename (before '_Day'), with a folder fallback."""
    stem = h5_path.stem
    if "_Day" in stem:
        return stem.split("_Day")[0]
    # fallback: session folder name like '20250825_mouse975826_trainingSessions'
    session = skeleton_folder.parent.parent.name
    for tok in session.split("_"):
        if tok.lower().startswith("mouse"):
            return tok
    return session  # last resort


def day_from(skeleton_folder: Path, h5_path: Path) -> str:
    """Day label for per_day layout: the Day folder name, or from filename."""
    day_name = skeleton_folder.parent.name
    if re.fullmatch(r"[Dd]ay\d+", day_name):
        return day_name
    m = re.search(r"(Day\d+)", h5_path.stem)
    return m.group(1) if m else day_name


def choose_method(session_folder: Path) -> str:
    flag = (session_folder / "flag.txt").exists()
    flag1 = (session_folder / "flag1.txt").exists()
    if flag1 or not flag:
        return "skeleton"
    return "transformer"  # flag and not flag1


def main() -> None:
    source = Path(SOURCE_ROOT)
    dest_root = Path(DEST_ROOT)
    if not source.exists():
        print(f"[ERROR] SOURCE_ROOT does not exist: {source}")
        sys.exit(1)

    skeleton_folders = sorted(
        p for p in source.rglob(f"*{PCUTOFF}_skeleton") if p.is_dir()
    )
    if not skeleton_folders:
        print(f"[ERROR] No '*{PCUTOFF}_skeleton' folders under {source}")
        sys.exit(1)

    print(f"{'DRY RUN — nothing will be copied' if DRY_RUN else 'COPYING'} "
          f"| layout={LAYOUT} | exclude_plot_poses={EXCLUDE_PLOT_POSES}\n")

    planned = 0
    copied = 0
    skipped = 0
    manifest = []  # (mouseID, day, method, chosen h5 in destination)

    for skel in skeleton_folders:
        session = skel.parent  # the Day folder (== parent.parent of the h5)

        skel_h5 = next(skel.glob(f"*{SKELETON_H5_SUFFIX}"), None)
        if skel_h5 is None:
            # No skeleton h5 -> this session isn't part of the dataset. Skip.
            continue

        method = choose_method(session)
        if method == "skeleton":
            chosen_folder = skel
            chosen_h5 = skel_h5
        else:
            chosen_folder = next(session.glob(f"*{PCUTOFF}_transformer"), None)
            chosen_h5 = (next(chosen_folder.glob(f"*{TRANSFORMER_H5_SUFFIX}"), None)
                         if chosen_folder else None)
            if chosen_folder is None or chosen_h5 is None:
                print(f"  [SKIP] {session}: flag.txt selects TRANSFORMER but its "
                      f"folder/h5 was not found.")
                skipped += 1
                continue

        mouse_id = mouse_id_from(skel_h5, skel)
        day = day_from(skel, skel_h5)

        if LAYOUT == "per_day":
            dst = dest_root / mouse_id / day / chosen_folder.name
        else:
            dst = dest_root / mouse_id / chosen_folder.name

        planned += 1
        dst_h5 = dst / chosen_h5.name
        manifest.append((mouse_id, day, method, str(dst_h5)))
        print(f"  [{method:11}] {mouse_id} {day}: "
              f"{chosen_folder.name}\n              -> {dst}")

        if not DRY_RUN:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(chosen_folder, dst, ignore=_IGNORE, dirs_exist_ok=True)
            copied += 1

    # Write a manifest of the selected h5 files (their DESTINATION paths).
    if not DRY_RUN and manifest:
        dest_root.mkdir(parents=True, exist_ok=True)
        man_path = dest_root / "selected_tracking_h5.txt"
        with man_path.open("w", encoding="utf-8") as f:
            for mouse_id, day, method, h5 in manifest:
                f.write(f"{mouse_id}\t{day}\t{method}\t{h5}\n")
        print(f"\nWrote manifest of {len(manifest)} selected h5 files: {man_path}")

    n_skel = sum(1 for m in manifest if m[2] == "skeleton")
    n_tr = sum(1 for m in manifest if m[2] == "transformer")
    print(f"\n{'Planned' if DRY_RUN else 'Copied'} {planned} session(s): "
          f"{n_skel} skeleton, {n_tr} transformer. Skipped: {skipped}.")
    if DRY_RUN:
        print("Review the list above, then set DRY_RUN = False to copy.")


if __name__ == "__main__":
    main()

COPYING | layout=merged | exclude_plot_poses=True

  [skeleton   ] mouse975826 Day02: topView_DLCtracking_pcutoff_0.8_skeleton
              -> C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions_topView_DLC_multiAnimal\trainingSessions_topView_DLC_multiAnimal\mouse975826\topView_DLCtracking_pcutoff_0.8_skeleton
  [transformer] mouse975826 Day03: topView_DLCtracking_pcutoff_0.8_transformer
              -> C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions_topView_DLC_multiAnimal\trainingSessions_topView_DLC_multiAnimal\mouse975826\topView_DLCtracking_pcutoff_0.8_transformer
  [skeleton   ] mouse975826 Day04: topView_DLCtracking_pcutoff_0.8_skeleton
              -> C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions_topView_DLC_multiAnimal\trainingSessions_topView_DLC_multiAnimal\mouse975826\topView_DLCtracking_pcutoff_0.8_skeleton
  [skeleton   ] mouse975826 Day05: topView_DLC